# Introduction to Vector Stores


- Use Langchain documentation to create vector stores using chromadb, milvus, weaviate, pinecone: https://python.langchain.com/docs/integrations/vectorstores/
- Create embeddings of a document and index into vector stores: https://huggingface.co/blog/getting-started-with-embeddings
- For the task you will be indexing the book: https://www.planetebook.com/free-ebooks/crime-and-punishment.pdf
- Experiment different vector stores and other variables for and present your findings for optimizing retrieval (the most appropriate parts should be returned for any query)
- Hint: chunking

**Steps**
- Go through the given readings. Take an hour at max.
- Index document as vectors. Use any documentation to help you, but avoid using AI Tools.
- For embedding model, use an open source model from huggingface.
- Query should return most appropriate parts in relevance to the query. *Remember, your task is to build effective retrieval*


**Resources for understanding vector search**
- https://weaviate.io/blog/vector-search-explained


**Additional resources for understanding embeddings**
- https://cohere.com/llmu/text-embeddings
- https://docs.cohere.com/v2/docs/embeddings
- https://docs.cohere.com/v2/docs/playground-overview

# **Install libraries**

# **Download the book**

In [1]:
!pip install -qU chromadb sentence-transformers pypdf requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.6/739.6 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.9/382.9 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.

In [2]:
import requests

pdf_url = "https://www.planetebook.com/free-ebooks/crime-and-punishment.pdf"
pdf_path = "/content/crime_and_punishment.pdf"

response = requests.get(pdf_url)

with open(pdf_path, "wb") as f:
    f.write(response.content)

print("PDF downloaded successfully.")
print("File size:", len(response.content), "bytes")

PDF downloaded successfully.
File size: 2488794 bytes


In [3]:
import os

print(os.path.exists(pdf_path))

True


# **Extract text from PDF**

In [4]:
from pypdf import PdfReader

reader = PdfReader(pdf_path)

print("Number of pages:", len(reader.pages))

Number of pages: 767


In [5]:
pages = []

for page_number, page in enumerate(reader.pages):
    text = page.extract_text()

    if text:
        pages.append({
            "page": page_number + 1,
            "text": text
        })

print("Pages with text:", len(pages))

Pages with text: 767


In [6]:
print(pages[0]["text"][:2000])

Download free eBooks of classic literature, books and 
novels at Planet eBook. Subscribe to our free eBooks blog 
and email newsletter.
Crime and Punishment
By Fyodor Dostoevsky



# **Combine the text**

In [7]:
full_text = "\n\n".join(
    page["text"] for page in pages
)

print("Total characters:", len(full_text))
print(full_text[:2000])

Total characters: 1173234
Download free eBooks of classic literature, books and 
novels at Planet eBook. Subscribe to our free eBooks blog 
and email newsletter.
Crime and Punishment
By Fyodor Dostoevsky


Crime and Punishment
Translator’s Preface
A 
few words about Dostoevsky himself may help the Eng -
lish reader to understand his work.
Dostoevsky was the son of a doctor. His parents were 
very hard- working and deeply religious people, but so poor 
that they lived with their five children in only two rooms. 
The father and mother spent their evenings in reading aloud 
to their children, generally from books of a serious charac -
ter.
Though always sickly and delicate Dostoevsky came out 
third in the final examination of the Petersburg school of 
Engineering. There he had already begun his first work, 
‘Poor Folk.’
This story was published by the poet Nekrassov in his 
review and was received with acclamations. The shy, un -
known youth found himself instantly something of a 
celeb

# **Clean the text**

In [8]:
import re

def clean_book_text(text):

    # Remove publisher/header/footer text
    text = re.sub(
        r'Free eBooks at Planet eBook\.com',
        ' ',
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r'Crime and Punishment',
        ' ',
        text,
        flags=re.IGNORECASE
    )

    # Remove weird non-printing characters
    text = re.sub(r'[^\x20-\x7E]+', ' ', text)

    # Fix words broken by hyphen + whitespace
    # e.g. "wait - ing" → "waiting"
    text = re.sub(r'(\w+)\s*-\s*(\w+)', r'\1\2', text)

    # Remove excessive whitespace
    text = re.sub(r'\s+', ' ', text)

    return text.strip()


clean_text = clean_book_text(full_text)

print(clean_text[:2000])

Download free eBooks of classic literature, books and novels at Planet eBook. Subscribe to our free eBooks blog and email newsletter. By Fyodor Dostoevsky Translator s Preface A few words about Dostoevsky himself may help the English reader to understand his work. Dostoevsky was the son of a doctor. His parents were very hardworking and deeply religious people, but so poor that they lived with their five children in only two rooms. The father and mother spent their evenings in reading aloud to their children, generally from books of a serious character. Though always sickly and delicate Dostoevsky came out third in the final examination of the Petersburg school of Engineering. There he had already begun his first work, Poor Folk. This story was published by the poet Nekrassov in his review and was received with acclamations. The shy, unknown youth found himself instantly something of a celebrity. A brilliant and successful career seemed to open before him, but those hopes were soon das

# **Chunking**

In [9]:
def create_chunks(text, chunk_size=1000, overlap=200):
    chunks = []

    start = 0

    while start < len(text):
        end = start + chunk_size

        chunk = text[start:end]

        if chunk.strip():
            chunks.append(chunk.strip())

        start += chunk_size - overlap

    return chunks

In [10]:
chunks = create_chunks(
    clean_text,
    chunk_size=1000,
    overlap=200
)

print("Number of chunks:", len(chunks))

Number of chunks: 1396


In [11]:
print(chunks[0])

Download free eBooks of classic literature, books and novels at Planet eBook. Subscribe to our free eBooks blog and email newsletter. By Fyodor Dostoevsky Translator s Preface A few words about Dostoevsky himself may help the English reader to understand his work. Dostoevsky was the son of a doctor. His parents were very hardworking and deeply religious people, but so poor that they lived with their five children in only two rooms. The father and mother spent their evenings in reading aloud to their children, generally from books of a serious character. Though always sickly and delicate Dostoevsky came out third in the final examination of the Petersburg school of Engineering. There he had already begun his first work, Poor Folk. This story was published by the poet Nekrassov in his review and was received with acclamations. The shy, unknown youth found himself instantly something of a celebrity. A brilliant and successful career seemed to open before him, but those hopes were soon das

In [12]:
print("\n--- CHUNK 2 ---\n")
print(chunks[1])


--- CHUNK 2 ---

and was received with acclamations. The shy, unknown youth found himself instantly something of a celebrity. A brilliant and successful career seemed to open before him, but those hopes were soon dashed. In 1849 he was arrested. Though neither by temperament nor conviction a revolutionist, Dostoevsky was one of a little group of young men who met together to read Fourier and Proudhon. He was accused of taking part in conversations against the censorship, of reading a letter from Byelinsky to Gogol, and of knowing of the intention to set up a printing press. Under Nicholas I. (that stern and just man, as Maurice Baring calls him) this was enough, and he was condemned to death. After eight months imprisonment he was with twentyone others taken out to the Semyonovsky Square to be shot. Writing to his brother Mihail, Dostoevsky says: They snapped words over our heads, and they made us put on the white shirts worn by persons condemned to death. Thereupon we were bound in t

# **Load the embedding model**

In [13]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.


# **Create embeddings**

In [14]:
embeddings = embedding_model.encode(
    chunks,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/44 [00:00<?, ?it/s]

Embedding shape: (1396, 384)


# **Create ChromaDB**

In [15]:
import chromadb

client = chromadb.PersistentClient(
    path="/content/crime_chroma_db"
)

collection = client.get_or_create_collection(
    name="crime_and_punishment"
)

print("ChromaDB collection created.")

ChromaDB collection created.


# **Store chunks and embeddings**

In [16]:
ids = [
    f"chunk_{i}"
    for i in range(len(chunks))
]

In [17]:
collection.add(
    ids=ids,
    documents=chunks,
    embeddings=embeddings.tolist(),
    metadatas=[
        {"chunk_id": i}
        for i in range(len(chunks))
    ]
)

print("Chunks indexed successfully.")
print("Total documents in vector store:", collection.count())

Chunks indexed successfully.
Total documents in vector store: 1396


# **Create the retrieval function**

In [18]:
def retrieve_documents(query, top_k=5):

    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )[0]

    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k
    )

    return results

# **Test your first query**

In [19]:
query = "Why does Raskolnikov feel guilty?"

results = retrieve_documents(query, top_k=5)

for i, document in enumerate(results["documents"][0]):

    distance = results["distances"][0][i]

    print(f"\n===== RESULT {i+1} =====")
    print("Distance:", distance)
    print(document)


===== RESULT 1 =====
Distance: 0.5753176808357239
nce I don t want to say anything against duty and conscience; but the point is, what do we mean by them. Stay, I have another question to ask you. Listen! No, you stay, I ll ask you a question. Listen! Well? You are talking and speechifying away, but tell me, would you kill the old woman yourself? Of course not! I was only arguing the justice of it . It s nothing to do with me . 101 But I think, if you would not do it yourself, there s no justice about it . Let us have another game. Raskolnikov was violently agitated. Of course, it was all ordinary youthful talk and thought, such as he had often heard before in different forms and on different themes. But why had he happened to hear such a discussion and such ideas at the very moment when his own brain was just conceiving the very same ideas? And why, just at the moment when he had brought away the embryo of his idea from the old woman had he dropped at once upon a conversation about h

# **Try multiple queries**

In [20]:
queries = [
    "Why does Raskolnikov commit the murder?",
    "What is Raskolnikov's relationship with Sonia?",
    "What role does poverty play in the story?",
    "Why does Raskolnikov confess?",
    "What are Raskolnikov's thoughts after the murder?"
]

In [21]:
for query in queries:

    print("\n" + "="*80)
    print("QUERY:", query)
    print("="*80)

    results = retrieve_documents(query, top_k=3)

    for i, document in enumerate(results["documents"][0]):

        print(f"\n--- Retrieved Passage {i+1} ---")
        print(document[:1000])


QUERY: Why does Raskolnikov commit the murder?

--- Retrieved Passage 1 ---
my sinister look at Raskolnikov. Murderer! he said suddenly in a quiet but clear and distinct voice. Raskolnikov went on walking beside him. His legs felt suddenly weak, a cold shiver ran down his spine, and his heart seemed to stand still for a moment, then suddenly began throbbing as though it were set free. So they walked for about a hundred paces, side by side in silence. The man did not look at him. What do you mean what is . Who is a murderer? muttered Raskolnikov hardly audibly. You are a murderer, the man answered still more articulately and emphatically, with a smile of triumphant hatred, and again he looked straight into Raskolnikov s pale face and stricken eyes. They had just reached the crossroads. The man turned to the left without looking behind him. Raskolnikov remained standing, gazing after him. He saw him turn round fifty paces away and look back at him still standing there. Raskolnikov could

# **Your first experiment: chunk size**

In [22]:
def build_collection(chunk_size, overlap, collection_name):

    chunks = create_chunks(
        clean_text,
        chunk_size=chunk_size,
        overlap=overlap
    )

    embeddings = embedding_model.encode(
        chunks,
        show_progress_bar=False,
        normalize_embeddings=True
    )

    collection = client.get_or_create_collection(
        name=collection_name
    )

    ids = [
        f"{collection_name}_{i}"
        for i in range(len(chunks))
    ]

    collection.add(
        ids=ids,
        documents=chunks,
        embeddings=embeddings.tolist()
    )

    return collection, chunks

In [23]:
collection_300, chunks_300 = build_collection(
    chunk_size=300,
    overlap=50,
    collection_name="chunks_300"
)

collection_500, chunks_500 = build_collection(
    chunk_size=500,
    overlap=100,
    collection_name="chunks_500"
)

collection_1000, chunks_1000 = build_collection(
    chunk_size=1000,
    overlap=200,
    collection_name="chunks_1000"
)

# **Compare retrieval**

In [24]:
test_query = "Why does Raskolnikov feel guilty?"

In [25]:
def search_collection(collection, query, top_k=3):

    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )[0]

    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k
    )

    return results

In [26]:
configs = {
    "300 chunks": collection_300,
    "500 chunks": collection_500,
    "1000 chunks": collection_1000
}

for name, collection in configs.items():

    print("\n")
    print("="*80)
    print(name)
    print("="*80)

    results = search_collection(
        collection,
        test_query,
        top_k=3
    )

    for i, document in enumerate(results["documents"][0]):

        print(f"\nResult {i+1}:")
        print(document[:500])



300 chunks

Result 1:
and refined with pretensions to gentility and smartness . The chief thing is, Raskolnikov persisted, to keep her out of this scoundrel s hands! Why should he outrage her! It s as clear as day what he is after; ah, the brute, he is not moving off! Raskolnikov spoke aloud and pointed to him. The gentl

Result 2:
just because he, Raskolnikov, had so hopelessly, stupidly come to grief through some decree of blind fate, and must humble himself and submit to the idiocy of a sentence, if he were anyhow to be at peace. Vague and objectless anxiety in the present, and in the future a continual sacrifice leading t

Result 3:
sence struck Raskolnikov unpleasantly. I ve got to think of that, he thought. Excuse me, please, he began, affecting extreme embarrassment. Raskolnikov. Not at all, very pleasant to see you and how pleasantly you ve come in . Why, won t he even say goodmorning? Porfiry Petrovitch nodded at Razumihin


500 chunks

Result 1:
f you would not do it yourse